<a href="https://colab.research.google.com/github/NathiJonas/Agentic-AI-Andela-Work/blob/main/sidekick_mj_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sidekick MJ — Agentic Personal Assistant
**Persona:** Master Jay (Jonas Thamane) | Data Engineer, South Africa  
**Stack:** LangGraph · Claude (claude-haiku-4-5) · Playwright · SendGrid · Pushover · Gradio 5.23.0

In [13]:
%pip install -q \
    langgraph \
    langchain-anthropic \
    langchain-core \
    playwright \
    sendgrid \
    requests \
    "gradio==5.23.0"

import subprocess
subprocess.run(["playwright", "install", "chromium", "--with-deps"], check=False)

CompletedProcess(args=['playwright', 'install', 'chromium', '--with-deps'], returncode=0)

In [10]:
import os

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
    SENDGRID_API_KEY  = userdata.get("SENDGRID_API_KEY")
    PUSHOVER_USER     = userdata.get("PUSHOVER_USER")
    PUSHOVER_TOKEN    = userdata.get("PUSHOVER_TOKEN")
    NOTIFY_EMAIL      = userdata.get("NOTIFY_EMAIL")
    FROM_EMAIL        = userdata.get("FROM_EMAIL")
    print("Secrets loaded from Colab userdata.")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
    SENDGRID_API_KEY  = os.environ.get("SENDGRID_API_KEY")
    PUSHOVER_USER     = os.environ.get("PUSHOVER_USER")
    PUSHOVER_TOKEN    = os.environ.get("PUSHOVER_TOKEN")
    NOTIFY_EMAIL      = os.environ.get("NOTIFY_EMAIL")
    FROM_EMAIL        = os.environ.get("FROM_EMAIL")
    print("Secrets loaded from environment variables.")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY or ""

Secrets loaded from Colab userdata.


In [3]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0.3)

In [4]:
!pip install langchain-anthropic anthropic


In [5]:

import requests
import json
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
from langchain_core.tools import tool
from playwright.async_api import async_playwright


@tool
async def scrape_webpage(url: str) -> str:
    """Fetch and return the visible text content of a web page."""
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await browser.new_page()
            await page.goto(url, timeout=15000)
            text = await page.inner_text("body")
            await browser.close()
        return text[:4000]  # trim to avoid token overload
    except Exception as e:
        return f"Error scraping {url}: {e}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email via SendGrid. Args: to (recipient), subject, body (plain text)."""
    try:
        message = Mail(
            from_email=FROM_EMAIL,
            to_emails=to,
            subject=subject,
            plain_text_content=body,
        )
        sg = SendGridAPIClient(SENDGRID_API_KEY)
        response = sg.send(message)
        return f"Email sent. Status: {response.status_code}"
    except Exception as e:
        return f"Email error: {e}"


@tool
def push_notification(message: str, title: str = "Sidekick MJ") -> str:
    """Send a push notification via Pushover. Args: message, title (optional)."""
    try:
        resp = requests.post(
            "https://api.pushover.net/1/messages.json",
            data={
                "token": PUSHOVER_TOKEN,
                "user": PUSHOVER_USER,
                "title": title,
                "message": message,
            },
            timeout=10,
        )
        return f"Pushover response: {resp.status_code} {resp.text}"
    except Exception as e:
        return f"Pushover error: {e}"


tools = [scrape_webpage, send_email, push_notification]
print(f"{len(tools)} tools registered: {[t.name for t in tools]}")

3 tools registered: ['scrape_webpage', 'send_email', 'push_notification']


In [6]:
!pip install playwright


In [7]:
!pip install sendgrid


In [8]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """\
You are Sidekick MJ, the personal AI assistant of Master Jay (Jonas Thamane),
a data engineer based in South Africa.

Your job is to help Master Jay with research, web lookups, scheduling reminders,
sending emails, and push notifications — whatever he needs.

Personality: sharp, efficient, occasionally witty. You speak directly and get
to the point. You know Master Jay well and adapt your tone to his style.

Tools available:
- scrape_webpage: fetch the text of any public URL
- send_email: send an email via SendGrid
- push_notification: send a Pushover alert to Master Jay's phone

Always confirm when actions (email/push) have been completed.
"""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("LangGraph ReAct agent ready.")

LangGraph ReAct agent ready.


/tmp/ipykernel_4522/694797654.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [12]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

conversation_history = []


async def chat(user_message: str, history: list):
    """Main chat handler — runs the LangGraph agent and streams the reply."""
    global conversation_history

    conversation_history.append(HumanMessage(content=user_message))


    result = await agent.ainvoke({"messages": conversation_history})
    messages = result["messages"]


    ai_reply = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            ai_reply = msg.content
            break


    conversation_history = messages
    return ai_reply


def reset_conversation():
    global conversation_history
    conversation_history = []
    return [], ""


with gr.Blocks(title="Sidekick MJ", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # Sidekick MJ
        **Personal AI Assistant — Master Jay (Jonas Thamane)**
        Powered by Claude (claude-haiku-4-5) · LangGraph · Playwright · SendGrid · Pushover
        """
    )

    chatbot = gr.Chatbot(
        label="Sidekick MJ",
        height=480,
        bubble_full_width=False,
    )

    with gr.Row():
        msg_box = gr.Textbox(
            placeholder="Ask Sidekick MJ anything...",
            show_label=False,
            scale=9,
        )
        send_btn = gr.Button("Send", scale=1, variant="primary")

    clear_btn = gr.Button("Clear conversation", variant="secondary")

    gr.Examples(
        examples=[
            "What are the top AI news stories today? Scrape https://techcrunch.com and summarise.",
            "Send me a push notification saying 'Heads up — stand-up in 5 minutes'",
            "Email me at " + (NOTIFY_EMAIL or "your@email.com") + " with a short motivational note.",
            "Summarise the Wikipedia page for LangGraph: https://en.wikipedia.org/wiki/LangChain",
        ],
        inputs=msg_box,
    )


    async def respond(user_msg, history):
        if not user_msg.strip():
            return history, ""
        reply = await chat(user_msg, history)
        history.append((user_msg, reply))
        return history, ""

    send_btn.click(respond, [msg_box, chatbot], [chatbot, msg_box])
    msg_box.submit(respond, [msg_box, chatbot], [chatbot, msg_box])
    clear_btn.click(reset_conversation, outputs=[chatbot, msg_box])


demo.launch(share=True)

/tmp/ipykernel_4522/460406013.py:48: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4522/460406013.py:48: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0fe77589f1828df84.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
